In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Silver Customers Validation").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
bronze_customers = spark.read.parquet("../data/bronze/customers")
silver_customers = spark.read.parquet("../data/silver/customers")

In [5]:
print("Bronze Customers:", bronze_customers.count())
print("Silver Customers:", silver_customers.count())

Bronze Customers: 99441
Silver Customers: 99441


In [13]:
from pyspark.sql.functions import count, col, when

In [11]:
duplicates_silver_customer = silver_customers.groupBy("customer_id").agg(
    count("customer_id").alias("count")
).filter(col("count")>1)

duplicates_silver_customer.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [14]:
silver_customers.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in silver_customers.columns
]).show(vertical=True)

-RECORD 0-----------------------
 customer_id              | 0   
 customer_unique_id       | 0   
 customer_zip_code_prefix | 0   
 customer_city            | 0   
 customer_state           | 0   



In [15]:
silver_customers.filter(
    col("customer_zip_code_prefix").startswith("0")
).show(10)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|208954ebe4665c5b2...|dc36c19e6e8bdcb2d...|                   09812|sao bernardo do c...|            SP|
|c2e1d65c47178eac5...|d1dae3a5d39b12618...|                   09370|                maua|            SP|
|167be5b45cbb0ea5a...|e3301490c373db926...|                   08223|           sao paulo|            SP|
|972cd773a42450ed3...|d7140656936a64175...|                   02013|           sao paulo|            SP|
|7b2a039a8b7b57d48...|fbf1b27fc02af440c...|                   02846|           sao paulo|            SP|
|e2691bbce4ba37372...|528079815d575acf6...|                   04045|           sao paulo|            SP|
|2f3dc6fda9c82bef0...|02a2a0a9627a06260...|            